# Phase 6A: Best Sequential Full Configuration
**Phase 6 | Final Best Setup Validation**
This notebook evaluates the ultimate combination found by sequentially picking the best components from Phases 1 through 5:
- Image: Swin-B | Text: PhoBERT | Fusion: Cross-Attention | Loss: Log-Cosh | Seed: 42


### STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### STEP 2: Clone source code and install dependencies

In [ ]:
!git clone https://github.com/lechihoang/SE365.git
%cd SE365
!pip install -r requirements.txt -q

Cloning into 'SE365'...
remote: Enumerating objects: 739, done.
remote: Counting objects: 100% (429/429), done.
remote: Compressing objects: 100% (180/180), done.
remote: Total 739 (delta 254), reused 419 (delta 249), pack-reused 310 (from 1)
Receiving objects: 100% (739/739), 14.47 MiB | 18.07 MiB/s, done.
Resolving deltas: 100% (463/463), done.
/content/SE365


### STEP 3: Download and extract data

In [ ]:
!rm -rf ./data
!cp /content/drive/MyDrive/SE365/data.zip ./data.zip
!unzip -q data.zip
!rm data.zip
!ls -la ./data

total 1420
drwxr-xr-x  4 root root    4096 Jun 16 09:21 .
drwxr-xr-x 11 root root    4096 Jun 24 16:12 ..
drwxr-xr-x  2 root root 1437696 Jun 16 09:59 image
drwxr-xr-x  2 root root    4096 Jun 16 09:21 text


### STEP 4: Configure paths

In [ ]:
import os
DRIVE_ROOT = '/content/drive/MyDrive/SE365'  # ✏️ Change if needed
EXP_ID = 'EXP_060A_bestsequential_full_configuration'

BEST_FUSION_EXP_ID = 'EXP_050C_bestfusion_logcosh'

DRIVE_EXP_PATH = f'{DRIVE_ROOT}/experiments/{EXP_ID}'
os.makedirs(DRIVE_EXP_PATH, exist_ok=True)
print(f'Artifacts: {DRIVE_EXP_PATH}')


Artifacts: /content/drive/MyDrive/SE365/experiments/EXP_060A_bestsequential_full_configuration


### STEP 5: Load pretrained weights

In [ ]:
import os
os.makedirs('./checkpoints', exist_ok=True)
!cp -r {DRIVE_ROOT}/experiments/{BEST_FUSION_EXP_ID}/* ./checkpoints/
print(f'Loaded all training artifacts and fusion model from {BEST_FUSION_EXP_ID}')


cp: target './experiments/EXP_060A_bestsequential_full_configuration/test_scatter_pred_vs_true.png' is not a directory
Loaded all training artifacts and fusion model from EXP_050C_bestfusion_logcosh


### STEP 6: Evaluate on Test Set
Evaluate the pre-trained model directly on the test set.

In [ ]:
!python test.py \
  --mode train_fusion \
  --fusion_type cross_attention \
  --text_model_name vinai/phobert-base-v2 \
  --image_model_name swin_base_patch4_window7_224 \
  --loss_fn logcosh \
  --exp_id $EXP_ID \
  --exp_dir ./experiments \
  --save_path ./checkpoints


====== TESTING: TRAIN_FUSION ======
Device: cuda
Test samples: 600
Loading weights: 100% 197/197 [00:00<00:00, 31919.88it/s]
[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loaded weights: ./checkpoints/best_model_train_fusion.pth
/content/SE365/test.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast

### STEP 7: Save to Drive + print metrics


In [ ]:
import json
!cp -r ./checkpoints/* $DRIVE_EXP_PATH/

# --- VALIDATION METRICS ---
with open('./checkpoints/metrics.json') as f:
    m = json.load(f)

print(f'\n=== {EXP_ID} Results (Validation) ===')
print(f"Loss (val)   : {m['loss']:.4f}")
print()
print("             MAE      RMSE      R2")
print(f"  food     : {m['mae_food']:.4f}   {m['rmse_food']:.4f}   {m['r2_food']:.4f}")
print(f"  price    : {m['mae_price']:.4f}   {m['rmse_price']:.4f}   {m['r2_price']:.4f}")
print(f"  atmos    : {m['mae_atmos']:.4f}   {m['rmse_atmos']:.4f}   {m['r2_atmos']:.4f}")
print(f"  service  : {m['mae_service']:.4f}   {m['rmse_service']:.4f}   {m['r2_service']:.4f}")
print(f"  overall  : {m['mae_overall']:.4f}   {m['rmse_overall']:.4f}   {m['r2_overall']:.4f}")
print()
print(f"  mean_mae   : {m['mean_mae']:.4f}")
print(f"  aspect_mae : {m['aspect_mae']:.4f}")
print(f"  overall_mae: {m['overall_mae']:.4f}")

# --- TEST METRICS ---
with open(f'./experiments/{EXP_ID}/test_metrics.json') as f:
    t = json.load(f)

print(f'\n=== {EXP_ID} Results (Test) ===')
print()
print("             MAE      RMSE      R2")
print(f"  food     : {t['mae_food']:.4f}   {t['rmse_food']:.4f}   {t['r2_food']:.4f}")
print(f"  price    : {t['mae_price']:.4f}   {t['rmse_price']:.4f}   {t['r2_price']:.4f}")
print(f"  atmos    : {t['mae_atmos']:.4f}   {t['rmse_atmos']:.4f}   {t['r2_atmos']:.4f}")
print(f"  service  : {t['mae_service']:.4f}   {t['rmse_service']:.4f}   {t['r2_service']:.4f}")
print(f"  overall  : {t['mae_overall']:.4f}   {t['rmse_overall']:.4f}   {t['r2_overall']:.4f}")
print()
print(f"  mean_mae   : {t['mean_mae']:.4f}")
print(f"  aspect_mae : {t['aspect_mae']:.4f}")
print(f"  overall_mae: {t['overall_mae']:.4f}")



=== EXP_060A_bestsequential_full_configuration Results (Validation) ===
Loss (val)   : 0.6413

             MAE      RMSE      R2
  food     : 1.1066   1.5006   0.5722
  price    : 1.1694   1.5671   0.4502
  atmos    : 1.1739   1.5250   0.4008
  service  : 1.1770   1.5697   0.5194
  overall  : 0.9130   1.2254   0.6312

  mean_mae   : 1.1080
  aspect_mae : 1.1567
  overall_mae: 0.9130

=== EXP_060A_bestsequential_full_configuration Results (Test) ===

             MAE      RMSE      R2
  food     : 1.0471   1.4692   0.6093
  price    : 1.1201   1.5015   0.4628
  atmos    : 1.1754   1.5565   0.3598
  service  : 1.0867   1.4890   0.5326
  overall  : 0.8819   1.1772   0.6479

  mean_mae   : 1.0622
  aspect_mae : 1.1073
  overall_mae: 0.8819
